# 🍇 Grape Disease Classification — MobileNetV3-Large

**Dataset:** Mendeley Data · DOI `10.17632/bsr2vzhrzr.2` · 2,197 grape images · 9 classes  
**Model:** MobileNetV3-Large (transfer learning, PyTorch)  
**Author:** Ranveersinh R. Patil Bhosale · COEP Technological University  
**Repo:** [Abhiagri10/grape-disease-detection](https://github.com/Abhiagri10/grape-disease-detection)

> ⚠️ **Runtime:** Set to **GPU (T4)** before running — Runtime → Change runtime type → T4 GPU  
> ▶️ Run cells **in order**. All code is self-contained — no external imports needed.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 1 — Install dependencies

In [ ]:
# Install only what Colab doesn't already have
import subprocess, sys

pkgs = ["pyyaml", "scikit-learn", "seaborn", "onnx", "onnxruntime", "tqdm"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

import torch, torchvision, sklearn
print(f"PyTorch     : {torch.__version__}")
print(f"Torchvision : {torchvision.__version__}")
print(f"CUDA        : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")


## Cell 2 — Configuration (all hyperparameters in one place)

In [ ]:
# ── All settings — edit here if needed ──────────────────────────────────────
import os

# Google Drive paths
DRIVE_ROOT   = "/content/drive/MyDrive/PlantDiseaseDataset/Custom Dataset for Plant Species and Disease Detec/"
ZIP_NAME     = "Ranveer_Realistic_dataset_2000_1126.zip"
ZIP_PATH     = os.path.join(DRIVE_ROOT, ZIP_NAME)
EXTRACT_DIR  = "/content/grape_only"          # local SSD — fast I/O
NESTED_PARENT = "Ranveer_Realistic_dataset_2000_1126"  # top folder inside zip

# Output dirs (saved to Drive — survives runtime restart)
OUT_ROOT     = os.path.join(DRIVE_ROOT, "outputs")
MODEL_DIR    = os.path.join(OUT_ROOT, "models")
PLOT_DIR     = os.path.join(OUT_ROOT, "plots")
REPORT_DIR   = os.path.join(OUT_ROOT, "reports")

# Dataset
SPECIES_PREFIX = "grape_"
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}
CLASS_NAMES = [
    "grape_anthracnose",
    "grape_guignardia_bidwellii",
    "grape_healthy",
    "grape_insect_eating",
    "grape_mineral_dificiency",          # intentional typo — matches dataset
    "grape_plasmopara_viticola",
    "grape_plasmopara_viticola_guignardia_bidwellii",
    "grape_plasmopara_viticola_insect_eating",
    "grape_powdery_mildew_insect_eating",
]
NUM_CLASSES = len(CLASS_NAMES)

# Model & training
IMG_SIZE      = 224
BATCH_SIZE    = 32
NUM_WORKERS   = 2
EPOCHS        = 40
LR            = 1e-3
WEIGHT_DECAY  = 1e-4
PATIENCE      = 8       # early stopping
LABEL_SMOOTH  = 0.1
SEED          = 42

# Split
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# Output file names
BEST_MODEL_NAME = "grape_mobilenetv3_best.pth"
ONNX_NAME       = "grape_mobilenetv3.onnx"

print(f"Classes    : {NUM_CLASSES}")
print(f"Image size : {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch size : {BATCH_SIZE}")
print(f"Epochs     : {EPOCHS}  (early stop patience={PATIENCE})")
print(f"Config ready ✓")


## Cell 3 — Reproducibility & device

In [ ]:
import random, numpy as np, torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"  GPU : {p.name}  |  VRAM: {p.total_memory/1e9:.1f} GB")
else:
    print("  ⚠️  No GPU — training will be very slow. Change runtime to T4 GPU.")


## Cell 4 — Mount Google Drive & verify zip

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
assert os.path.exists(ZIP_PATH), (
    f"\n❌  ZIP not found: {ZIP_PATH}"
    f"\n\nFix: Upload '{ZIP_NAME}' to Google Drive at:"
    f"\n     /content/drive/MyDrive/PlantDiseaseDataset/Custom Dataset for Plant Species and Disease Detec/"
)

size_gb = os.path.getsize(ZIP_PATH) / 1e9
print(f"✅  ZIP found: {ZIP_PATH}")
print(f"    Size    : {size_gb:.2f} GB")

# Create output dirs on Drive
for d in [MODEL_DIR, PLOT_DIR, REPORT_DIR]:
    os.makedirs(d, exist_ok=True)
print(f"✅  Output dirs ready at {OUT_ROOT}")


## Cell 5 — Extract grape-only folders from zip

In [ ]:
import zipfile
from pathlib import Path
from tqdm import tqdm

def extract_grape_only(zip_path, extract_dir, species_prefix=SPECIES_PREFIX,
                       nested_parent=NESTED_PARENT, skip_if_exists=True):
    """
    Extract only grape_* folders. Handles nested zip structure:
      Ranveer_Realistic_dataset_2000_1126/grape_healthy/img.jpg
    Strips the nested parent so output is flat:
      /content/grape_only/grape_healthy/img.jpg
    """
    ep = Path(extract_dir)

    # Skip if already done
    if skip_if_exists and ep.exists():
        existing = [p for p in ep.iterdir() if p.is_dir() and p.name.startswith(species_prefix)]
        if existing:
            print(f"⏭  Skipped — {len(existing)} grape folders already extracted")
            return

    ep.mkdir(parents=True, exist_ok=True)
    prefix = f"{nested_parent}/{species_prefix}" if nested_parent else species_prefix

    with zipfile.ZipFile(zip_path, "r") as zf:
        all_m  = zf.namelist()
        grape_m = [m for m in all_m if prefix in m and not m.endswith("/")]

        print(f"Total zip entries : {len(all_m):,}")
        print(f"Grape file entries: {len(grape_m):,}")

        if not grape_m:
            sample = all_m[:8]
            raise ValueError(
                f"No entries matching prefix '{prefix}' found.\n"
                f"First 8 zip entries:\n" + "\n".join(f"  {e}" for e in sample) +
                f"\n\nCheck NESTED_PARENT in Cell 2."
            )

        for member in tqdm(grape_m, desc="Extracting"):
            rel = member[len(nested_parent)+1:] if nested_parent and member.startswith(nested_parent+"/") else member
            dest = ep / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(member) as src, open(dest, "wb") as dst:
                dst.write(src.read())

    found = [p.name for p in ep.iterdir() if p.is_dir() and p.name.startswith(species_prefix)]
    print(f"\n✅  Extracted {len(found)} grape class folders to {extract_dir}")

extract_grape_only(ZIP_PATH, EXTRACT_DIR)

# Verify all 9 class folders exist
missing = [c for c in CLASS_NAMES if not os.path.isdir(os.path.join(EXTRACT_DIR, c))]
if missing:
    found_dirs = sorted(os.listdir(EXTRACT_DIR))
    raise FileNotFoundError(
        f"Missing class folders: {missing}\n"
        f"Actual contents: {found_dirs[:15]}"
    )
print(f"✅  All {NUM_CLASSES} class folders verified")


## Cell 6 — Inspect class distribution

In [ ]:
from collections import Counter
from pathlib import Path

def collect_samples(data_dir, class_names, exts):
    samples = []
    for idx, cls in enumerate(class_names):
        cls_dir = Path(data_dir) / cls
        for fp in sorted(cls_dir.iterdir()):
            if fp.suffix.lower() in exts:
                samples.append((str(fp), idx))
    return samples

all_samples = collect_samples(EXTRACT_DIR, CLASS_NAMES, IMG_EXTS)

counts = Counter(s[1] for s in all_samples)
total  = len(all_samples)
print(f"{'='*60}")
print(f"  Class Distribution — {total} total images")
print(f"{'='*60}")
print(f"  {'Class':<50} {'N':>4}  {'%':>5}")
print(f"  {'-'*58}")
for idx, name in enumerate(CLASS_NAMES):
    n = counts[idx]
    bar = "█" * int(20 * n / max(counts.values()))
    print(f"  {name:<50} {n:>4}  {100*n/total:>4.1f}%  {bar}")
print(f"  {'-'*58}")
print(f"  {'TOTAL':<50} {total:>4}")

assert total > 0, "No images found"
assert len(set(s[1] for s in all_samples)) == NUM_CLASSES, "Some classes have 0 images"
print("\n✅  Distribution verified")


## Cell 7 — Stratified train / val / test split

In [ ]:
from sklearn.model_selection import train_test_split

labels = [s[1] for s in all_samples]

# Step 1: train vs (val+test)
temp_ratio = VAL_RATIO + TEST_RATIO
train_s, temp_s, _, temp_labels = train_test_split(
    all_samples, labels,
    test_size=temp_ratio, stratify=labels, random_state=SEED
)

# Step 2: val vs test
val_frac = VAL_RATIO / temp_ratio
val_s, test_s = train_test_split(
    temp_s, test_size=(1 - val_frac),
    stratify=temp_labels, random_state=SEED
)

steps_train = len(train_s) // BATCH_SIZE
steps_val   = -(-len(val_s) // BATCH_SIZE)

print(f"{'='*55}")
print(f"  {'Class':<44} {'Tr':>4} {'Va':>4} {'Te':>4}")
print(f"  {'-'*53}")
for idx, name in enumerate(CLASS_NAMES):
    nt = sum(1 for s in train_s if s[1]==idx)
    nv = sum(1 for s in val_s   if s[1]==idx)
    ne = sum(1 for s in test_s  if s[1]==idx)
    print(f"  {name:<44} {nt:>4} {nv:>4} {ne:>4}")
print(f"  {'-'*53}")
print(f"  {'TOTAL':<44} {len(train_s):>4} {len(val_s):>4} {len(test_s):>4}")
print(f"  Ratio : {len(train_s)/total*100:.1f}% / {len(val_s)/total*100:.1f}% / {len(test_s)/total*100:.1f}%")
print(f"  Steps/epoch  Train:{steps_train}  Val:{steps_val}")


## Cell 8 — Preprocessing, augmentation & DataLoaders

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, UnidentifiedImageError

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.2),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class GrapeDiseaseDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert("RGB")
        except (UnidentifiedImageError, OSError):
            img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), 0)
        return self.transform(img), label

loaders = {
    "train": DataLoader(GrapeDiseaseDataset(train_s, train_tf),
                        batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True),
    "val":   DataLoader(GrapeDiseaseDataset(val_s,   eval_tf),
                        batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True),
    "test":  DataLoader(GrapeDiseaseDataset(test_s,  eval_tf),
                        batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True),
}

# Batch shape sanity check
imgs, lbls = next(iter(loaders["train"]))
assert imgs.shape == (BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE), f"Unexpected shape: {imgs.shape}"
print(f"Train batch : {imgs.shape}  dtype={imgs.dtype}")
print(f"Labels sample: {lbls[:8].tolist()}")
print("✅  DataLoaders ready")


## Cell 9 — Visualise augmented samples (sanity check)

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np, os

mean = np.array(IMAGENET_MEAN)
std  = np.array(IMAGENET_STD)

def denorm(t):
    img = t.permute(1, 2, 0).numpy() * std + mean
    return np.clip(img, 0, 1)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(denorm(imgs[i]))
    ax.set_title(CLASS_NAMES[lbls[i]].replace("grape_",""), fontsize=7)
    ax.axis("off")
plt.suptitle("Augmented Training Samples", fontsize=12)
plt.tight_layout()

save_path = os.path.join(PLOT_DIR, "sample_batch.png")
fig.savefig(save_path, dpi=120, bbox_inches="tight")
print(f"Saved → {save_path}")
plt.show()


## Cell 10 — Build MobileNetV3-Large (transfer learning)

In [ ]:
import torch.nn as nn
import torchvision.models as models
from torchvision.models import MobileNet_V3_Large_Weights

def build_model(num_classes):
    model = models.mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1)

    # Freeze entire backbone
    for p in model.features.parameters():
        p.requires_grad = False

    # Unfreeze last 3 feature blocks for fine-tuning
    for block in model.features[-3:]:
        for p in block.parameters():
            p.requires_grad = True

    # Replace classifier head
    in_f = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_f, num_classes)
    return model

model = build_model(NUM_CLASSES).to(DEVICE)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Architecture  : MobileNetV3-Large")
print(f"Output classes: {NUM_CLASSES}")
print(f"Total params  : {total:,}")
print(f"Trainable     : {trainable:,}  ({100*trainable/total:.1f}%)")

# Forward pass check
with torch.no_grad():
    dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    out   = model(dummy)
assert out.shape == (2, NUM_CLASSES), f"Bad output shape: {out.shape}"
print(f"Forward pass  : {out.shape} ✓")


## Cell 11 — Train  *(~15–25 min on T4)*

In [ ]:
import time, os
from torch.amp import GradScaler, autocast
from pathlib import Path

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH).to(DEVICE)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = torch.amp.GradScaler('cuda')

best_ckpt = os.path.join(MODEL_DIR, BEST_MODEL_NAME)
Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)

history = {"train_loss":[], "train_acc":[], "val_loss":[], "val_acc":[]}
best_val_acc   = 0.0
patience_count = 0

print(f"{'Ep':>4}  {'Tr Loss':>8}  {'Tr Acc':>7}  {'Va Loss':>8}  {'Va Acc':>7}  {'LR':>9}  {'Time':>6}")
print("─" * 65)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # ── Train ──
    model.train()
    tr_loss = tr_correct = tr_total = 0
    for imgs_b, labs_b in loaders["train"]:
        imgs_b, labs_b = imgs_b.to(DEVICE, non_blocking=True), labs_b.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            out  = model(imgs_b)
            loss = criterion(out, labs_b)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        tr_loss    += loss.item() * imgs_b.size(0)
        tr_correct += out.argmax(1).eq(labs_b).sum().item()
        tr_total   += labs_b.size(0)

    # ── Validate ──
    model.eval()
    va_loss = va_correct = va_total = 0
    with torch.no_grad():
        for imgs_b, labs_b in loaders["val"]:
            imgs_b, labs_b = imgs_b.to(DEVICE, non_blocking=True), labs_b.to(DEVICE, non_blocking=True)
            with torch.amp.autocast('cuda'):
                out  = model(imgs_b)
                loss = criterion(out, labs_b)
            va_loss    += loss.item() * imgs_b.size(0)
            va_correct += out.argmax(1).eq(labs_b).sum().item()
            va_total   += labs_b.size(0)

    scheduler.step()

    tr_l, tr_a = tr_loss/tr_total, tr_correct/tr_total
    va_l, va_a = va_loss/va_total, va_correct/va_total
    lr_now     = scheduler.get_last_lr()[0]
    elapsed    = time.time() - t0

    history["train_loss"].append(tr_l)
    history["train_acc"].append(tr_a)
    history["val_loss"].append(va_l)
    history["val_acc"].append(va_a)

    marker = ""
    if va_a > best_val_acc:
        best_val_acc   = va_a
        patience_count = 0
        torch.save({
            "epoch": epoch, "best_val_acc": best_val_acc,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
        }, best_ckpt)
        marker = "  ✓ saved"
    else:
        patience_count += 1

    print(f"{epoch:>4}  {tr_l:>8.4f}  {tr_a:>7.4f}  {va_l:>8.4f}  {va_a:>7.4f}  {lr_now:>9.2e}  {elapsed:>5.1f}s{marker}")

    if patience_count >= PATIENCE:
        print(f"\nEarly stop at epoch {epoch} (no improvement for {PATIENCE} epochs)")
        break

print(f"\n✅  Training complete — best val acc: {best_val_acc:.4f}")
print(f"   Checkpoint: {best_ckpt}")


## Cell 12 — Plot training curves

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker, os

epochs_ran = range(1, len(history["train_loss"]) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(epochs_ran, history["train_loss"], label="Train", lw=1.8)
ax1.plot(epochs_ran, history["val_loss"],   label="Val",   lw=1.8)
ax1.set(title="Loss", xlabel="Epoch", ylabel="Loss"); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs_ran, history["train_acc"], label="Train", lw=1.8)
ax2.plot(epochs_ran, history["val_acc"],   label="Val",   lw=1.8)
ax2.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1.0))
ax2.set(title="Accuracy", xlabel="Epoch", ylabel="Accuracy"); ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle("MobileNetV3-Large — Grape Disease Detection", fontsize=13, y=1.01)
plt.tight_layout()

save_path = os.path.join(PLOT_DIR, "training_curves.png")
fig.savefig(save_path, dpi=150, bbox_inches="tight")
print(f"Saved → {save_path}")
plt.show()


## Cell 13 — Load best checkpoint for evaluation

In [ ]:
# Re-build a clean model and load the best weights
model_eval = build_model(NUM_CLASSES).to(DEVICE)
ckpt = torch.load(best_ckpt, map_location=DEVICE)
model_eval.load_state_dict(ckpt["model_state_dict"])
model_eval.eval()

print(f"Loaded checkpoint from epoch {ckpt['epoch']}")
print(f"Best val acc : {ckpt['best_val_acc']:.4f}  ({100*ckpt['best_val_acc']:.2f}%)")


## Cell 14 — Test set evaluation

In [ ]:
import numpy as np
from torch.cuda.amp import autocast

all_preds  = []
all_labels = []

model_eval.eval()
with torch.no_grad():
    for imgs_b, labs_b in loaders["test"]:
        imgs_b = imgs_b.to(DEVICE, non_blocking=True)
        with torch.amp.autocast('cuda'):
            out = model_eval(imgs_b)
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(labs_b.numpy())

y_pred = np.array(all_preds)
y_true = np.array(all_labels)
test_acc = (y_pred == y_true).mean()

print(f"Test samples : {len(y_true)}")
print(f"Test Accuracy: {test_acc:.4f}  ({100*test_acc:.2f}%)")


## Cell 15 — Classification report

In [ ]:
from sklearn.metrics import classification_report
import os

short = [c.replace("grape_","") for c in CLASS_NAMES]
report = classification_report(y_true, y_pred, target_names=short, digits=4)
print(report)

save_path = os.path.join(REPORT_DIR, "classification_report.txt")
with open(save_path, "w") as f:
    f.write(f"Test Accuracy: {test_acc:.4f}\n\n" + report)
print(f"Saved → {save_path}")


## Cell 16 — Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns, matplotlib.pyplot as plt, os

short = [c.replace("grape_","") for c in CLASS_NAMES]
cm    = confusion_matrix(y_true, y_pred)
cm_n  = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm_n, annot=True, fmt=".2f",
            xticklabels=short, yticklabels=short,
            cmap="Blues", linewidths=0.4, ax=ax)
ax.set_xlabel("Predicted", fontsize=11)
ax.set_ylabel("True", fontsize=11)
ax.set_title(f"Confusion Matrix (Row-Normalized)  —  Test Acc: {test_acc:.4f}", fontsize=12)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()

save_path = os.path.join(PLOT_DIR, "confusion_matrix.png")
fig.savefig(save_path, dpi=150, bbox_inches="tight")
print(f"Saved → {save_path}")
plt.show()


## Cell 17 — Export to ONNX

In [ ]:
import onnx, os
from pathlib import Path

# Install onnxscript if it's missing (it's a dependency for torch.onnx.export in some PyTorch versions)
try:
    import onnxscript
except ImportError:
    %pip install -q onnxscript

onnx_path = os.path.join(MODEL_DIR, ONNX_NAME)
model_eval.eval().cpu()

dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
torch.onnx.export(
    model_eval, dummy, onnx_path,
    export_params=True, opset_version=18,
    do_constant_folding=True,
    input_names=["input"], output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
)

onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print(f"✅  ONNX export verified: {onnx_path}")
print(f"    Size: {os.path.getsize(onnx_path)/1e6:.1f} MB")

# Move model back to device for any further use
model_eval.to(DEVICE)


## Cell 18 — ONNX inference sanity check

In [ ]:
import onnxruntime as ort
import numpy as np

sess    = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
dummy_n = np.random.randn(1, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)
out_n   = sess.run(None, {"input": dummy_n})[0]
pred_i  = int(out_n.argmax())

print(f"ONNX output shape    : {out_n.shape}")
print(f"Sample prediction    : {CLASS_NAMES[pred_i]}")
print(f"✅  ONNX runtime inference working")


## Cell 19 — Final summary

In [ ]:
import os

pth_mb  = os.path.getsize(best_ckpt) / 1e6
onnx_mb = os.path.getsize(onnx_path) / 1e6

print("=" * 60)
print("  PIPELINE COMPLETE")
print("=" * 60)
print(f"  Model        : MobileNetV3-Large")
print(f"  Classes      : {NUM_CLASSES}")
print(f"  Train images : {len(train_s)}")
print(f"  Val images   : {len(val_s)}")
print(f"  Test images  : {len(test_s)}")
print(f"  Best val acc : {ckpt['best_val_acc']:.4f}  ({100*ckpt['best_val_acc']:.2f}%)")
print(f"  Test acc     : {test_acc:.4f}  ({100*test_acc:.2f}%)")
print("=" * 60)
print(f"\nSaved to Drive ({OUT_ROOT}):")
print(f"  models/{BEST_MODEL_NAME}  ({pth_mb:.1f} MB)")
print(f"  models/{ONNX_NAME}         ({onnx_mb:.1f} MB)")
print(f"  plots/training_curves.png")
print(f"  plots/confusion_matrix.png")
print(f"  plots/sample_batch.png")
print(f"  reports/classification_report.txt")
